# NB40 — Graph-Based Routing Proof (Phase 1.5)## What this notebook provesThe old routing code (`get_next_destination`) walks the role list top-to-bottomand returns the **first facility it finds** with a matching edge. When two R1nodes exist in parallel, the second one never gets picked — dict iteration orderdecides who wins, not the network topology.Phase 1.5 replaces that loop with a **Dijkstra shortest-path query** throughthe existing `TreatmentNetwork` graph. Edge weights are updated dynamicallybased on facility occupancy, so routes shift toward less-loaded nodes.### How it works (three moving parts)1. **`_find_highest_reachable()`** — walks ROLE_ORDER from R4 down to the   current role, returns the first facility reachable via `nx.has_path()`.   This is the Dijkstra *target*. For bypass patients, R1 nodes are excluded   from the reachability graph.2. **`network.get_route(patient, from, to)`** — calls `nx.dijkstra_path()`   using dynamic `weight` attributes on edges. Returns the full path;   routing takes `path[1]` as the next hop.3. **`_update_facility_congestion()`** — called in `engine.py` at every   `FACILITY_ARRIVAL` and `DISPOSITION`. Computes `occupancy / capacity`   and pushes it into inbound edge weights via `update_congestion()`.   This makes full facilities "heavier" so Dijkstra steers around them.### Toggle gateEverything is behind `enable_graph_routing` (default `False`). Toggle OFF =legacy first-match behavior unchanged. Toggle ON = Dijkstra with congestion.Linear backward compat (toggle OFF == ON on single-path topologies) isvalidated by 44 pytest assertions — this notebook focuses on multi-path proof.### Exit criteria (ALL must pass)| # | Criterion | What it checks ||---|-----------|----------------|| 1 | Both R1 nodes active | IRON BRIDGE 48h, R1-BRAVO gets >0 traffic || 2 | Congestion shifts route | Heavy load on R1-A pushes Dijkstra to R1-B || 3 | HC-2 determinism | Same seed = same output with graph routing || 4 | KL-6 DISPOSITION invariant | arrivals == dispositions + in_system |

---## Cell 1: Imports + Setup

In [ ]:
import sys, os# Resolve src/ whether CWD is project root or notebooks/phase2/for _candidate in [    os.path.join(os.path.abspath('.'), 'src'),    os.path.join(os.path.abspath('..'), '..', 'src'),    os.path.join(os.path.abspath('..'), 'src'),]:    if os.path.isdir(_candidate) and _candidate not in sys.path:        sys.path.insert(0, _candidate)        breakfrom collections import Counterimport matplotlib.pyplot as pltimport matplotlib.patches as mpatchesimport networkx as nxfrom faer_dev.config.builder import build_engine_from_presetfrom faer_dev.decisions.mode import SimulationTogglesfrom faer_dev.core.enums import Role, TriageCategoryfrom faer_dev.core.schemas import Casualty, Facilityfrom faer_dev.network.topology import TreatmentNetworkfrom faer_dev.routing import get_next_destination, triage_decisionsGRAPH_ON = SimulationToggles(    enable_extracted_routing=True,    enable_graph_routing=True,)ROLE_COLORS = {    Role.POI: "#e74c3c",    Role.R1: "#f39c12",    Role.R2: "#2ecc71",    Role.R3: "#3498db",    Role.R4: "#9b59b6",}def run_iron_bridge(seed=42, duration=2880.0, max_patients=None):    engine = build_engine_from_preset("iron_bridge", seed=seed, toggles=GRAPH_ON)    metrics = engine.run(duration=duration, max_patients=max_patients)    return metrics, engineprint("Imports OK")

---## IRON BRIDGE TopologyFive nodes, two parallel R1 facilities. Both edges from POI have identicalbase_time (15 min) and identical bed counts (8). Without congestion-awarerouting, only the first one in dict order gets traffic.

In [ ]:
def draw_topology(ax, graph, facilities, title, pos_override=None):    pos = pos_override or {}    if not pos:        for fid, fac in facilities.items():            pos[fid] = fac.coordinates    colors = [ROLE_COLORS.get(facilities[n].role, "#999") for n in graph.nodes()]    labels = {}    for n in graph.nodes():        fac = facilities[n]        labels[n] = f"{fac.id}\n({fac.role.name}, {fac.beds}b)"    nx.draw_networkx_nodes(graph, pos, ax=ax, node_color=colors,                           node_size=2200, edgecolors="black", linewidths=1.5)    nx.draw_networkx_labels(graph, pos, labels=labels, ax=ax, font_size=8,                            font_weight="bold")    edge_labels = {}    for u, v, d in graph.edges(data=True):        edge_labels[(u, v)] = f"{d['base_time']:.0f}m"    nx.draw_networkx_edges(graph, pos, ax=ax, edge_color="#555",                           width=2, arrows=True, arrowsize=20,                           connectionstyle="arc3,rad=0.1")    nx.draw_networkx_edge_labels(graph, pos, edge_labels=edge_labels, ax=ax,                                 font_size=9, font_color="#333")    ax.set_title(title, fontsize=13, fontweight="bold", pad=12)    ax.axis("off")# Build and draw IRON BRIDGEib_engine_viz = build_engine_from_preset("iron_bridge", seed=42, toggles=GRAPH_ON)ib_net = ib_engine_viz.networkib_pos = {    "POI-FRONT": (0, 0),    "R1-ALPHA": (2, 1.2),    "R1-BRAVO": (2, -1.2),    "R2-MAIN": (4, 0),    "R3-MAIN": (7, 0),}fig, ax = plt.subplots(figsize=(10, 5))draw_topology(ax, ib_net.graph, ib_net.facilities,              "IRON BRIDGE — 5-Node LSCO Topology", ib_pos)patches = [mpatches.Patch(color=c, label=r.name) for r, c in ROLE_COLORS.items()           if r in (Role.POI, Role.R1, Role.R2, Role.R3)]ax.legend(handles=patches, loc="lower right", fontsize=9, frameon=True)plt.tight_layout()plt.show()print("Edge labels = base_time in minutes (before congestion).")

---## Criterion 1: IRON BRIDGE 48h — Both R1 Nodes Get TrafficThe whole point of graph routing: with congestion wired, Dijkstra shiftstraffic from overloaded R1-ALPHA toward R1-BRAVO. Both should receivepatients, with an imbalance ratio below 0.30.

In [ ]:
# Run IRON BRIDGE 48h with graph routingm_ib, engine_ib = run_iron_bridge(seed=42)def count_facility_arrivals(engine):    arrivals = engine.event_store.events_of_type("FACILITY_ARRIVAL")    return Counter(e.facility_id for e in arrivals)graph_counts = count_facility_arrivals(engine_ib)# Summary tablefac_ids = ["R1-ALPHA", "R1-BRAVO", "R2-MAIN", "R3-MAIN"]print(f"{'Facility':<14} {'Arrivals':<10} {'Beds':<6}")print("-" * 32)for fid in fac_ids:    beds = engine_ib.network.facilities[fid].beds    print(f"{fid:<14} {graph_counts.get(fid, 0):<10} {beds:<6}")print(f"\nTotal arrivals: {m_ib['total_arrivals']}")print(f"Completed: {m_ib['completed']}, In system: {m_ib['in_system']}")print(f"Outcomes: {m_ib['outcomes']}")# Bar chartfig, ax = plt.subplots(figsize=(8, 5))vals = [graph_counts.get(f, 0) for f in fac_ids]colors = [ROLE_COLORS.get(engine_ib.network.facilities[f].role, "#999") for f in fac_ids]bars = ax.bar(fac_ids, vals, color=colors, edgecolor="black", alpha=0.9)for bar in bars:    h = bar.get_height()    if h > 0:        ax.text(bar.get_x() + bar.get_width()/2, h + 3, str(int(h)),                ha="center", fontsize=10, fontweight="bold")ax.set_ylabel("FACILITY_ARRIVAL count")ax.set_title("IRON BRIDGE 48h — Facility Arrivals (Graph Routing)", fontweight="bold")ax.spines["top"].set_visible(False)ax.spines["right"].set_visible(False)plt.tight_layout()plt.show()# Assertionsr1a = graph_counts.get("R1-ALPHA", 0)r1b = graph_counts.get("R1-BRAVO", 0)total_r1 = r1a + r1bimbalance = abs(r1a - r1b) / total_r1 if total_r1 > 0 else 1.0assert r1a > 0, "R1-ALPHA got zero traffic"assert r1b > 0, "R1-BRAVO got zero traffic — first-match bias NOT fixed"assert imbalance < 0.30, f"Imbalance {imbalance:.3f} > 0.30"print(f"\nR1 balance: ALPHA={r1a}, BRAVO={r1b}, imbalance={imbalance:.3f}")print("\n[PASS] Both R1 nodes active and balanced")

### Arrivals over time — cumulative R1 loadHow traffic accumulates at each R1 node over the 48h sim.Both curves should climb together. ALPHA leads slightly because it winsDijkstra tiebreaks at equal weights, but BRAVO stays close as congestionpushes back.

In [ ]:
arrivals = engine_ib.event_store.events_of_type("FACILITY_ARRIVAL")alpha_times = sorted(e.sim_time for e in arrivals if e.facility_id == "R1-ALPHA")bravo_times = sorted(e.sim_time for e in arrivals if e.facility_id == "R1-BRAVO")fig, ax = plt.subplots(figsize=(10, 5))ax.step([t/60 for t in alpha_times], range(1, len(alpha_times)+1), where="post",        color="#f39c12", linewidth=2, label=f"R1-ALPHA ({len(alpha_times)})")ax.step([t/60 for t in bravo_times], range(1, len(bravo_times)+1), where="post",        color="#e67e22", linewidth=2, linestyle="--", label=f"R1-BRAVO ({len(bravo_times)})")ax.set_xlabel("Simulation time (hours)")ax.set_ylabel("Cumulative arrivals")ax.set_title("Cumulative R1 Arrivals — IRON BRIDGE 48h", fontweight="bold")ax.legend(loc="upper left", fontsize=10)ax.spines["top"].set_visible(False)ax.spines["right"].set_visible(False)plt.tight_layout()plt.show()

---## Criterion 2: Congestion Shifts RouteDijkstra alone isn't enough — with equal edge weights, it'll still preferwhichever node sorts first. The real mechanism is **dynamic congestion**.When a patient arrives at a facility, the engine computes:```congestion_factor = queue.count / queue.capacity```Then calls `network.update_congestion(facility_id, factor)`, which setsevery inbound edge's weight to:```weight = base_time * (1 + congestion_factor)```A facility at 100% capacity doubles its inbound edge weights.A facility at 200% capacity triples them. This makes full facilities"expensive" for Dijkstra to route into.### Unit proofWe build a symmetric 4-node network, then sweep congestion on R1-A from0.0 to 9.0. At some point Dijkstra flips to R1-B.

In [ ]:
# Build symmetric 4-node networknet = TreatmentNetwork()net.add_facility(Facility(id="POI", name="POI", role=Role.POI, beds=0))net.add_facility(Facility(id="R1-A", name="R1-A", role=Role.R1, beds=4))net.add_facility(Facility(id="R1-B", name="R1-B", role=Role.R1, beds=4))net.add_facility(Facility(id="R2", name="R2", role=Role.R2, beds=8))net.add_route("POI", "R1-A", base_time=30.0)net.add_route("POI", "R1-B", base_time=30.0)net.add_route("R1-A", "R2", base_time=45.0)net.add_route("R1-B", "R2", base_time=45.0)patient = Casualty(    id="TEST-T2", name="Test", triage=TriageCategory.T2,    initial_triage=TriageCategory.T2, created_at=0.0, state_changed_at=0.0,)decisions = triage_decisions(patient)poi = net.facilities["POI"]# Sweep congestion factorsprint(f"{'Congestion on R1-A':<22} {'POI->R1-A weight':<18} {'POI->R1-B weight':<18} {'Next hop':>10}")print("-" * 70)test_factors = [0.0, 0.5, 1.0, 2.0, 5.0, 9.0]results = []for factor in test_factors:    net.update_congestion("R1-A", 0.0)    net.update_congestion("R1-B", 0.0)    net.update_congestion("R1-A", factor)    w_a = net.graph["POI"]["R1-A"]["weight"]    w_b = net.graph["POI"]["R1-B"]["weight"]    dest = get_next_destination(patient, poi, net, decisions, use_graph_routing=True)    results.append((factor, w_a, w_b, dest))    print(f"factor={factor:<16.1f} {w_a:<18.1f} {w_b:<18.1f} {dest:>10}")assert results[-1][3] == "R1-B", f"Expected R1-B at factor=9.0, got {results[-1][3]}"# Visualsfig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))factors = [r[0] for r in results]weights_a = [r[1] for r in results]weights_b = [r[2] for r in results]hops = [r[3] for r in results]ax1.plot(factors, weights_a, "o-", color="#e74c3c", linewidth=2, markersize=8, label="POI -> R1-A")ax1.plot(factors, weights_b, "s--", color="#2ecc71", linewidth=2, markersize=8, label="POI -> R1-B")ax1.set_xlabel("Congestion factor on R1-A")ax1.set_ylabel("Edge weight (minutes)")ax1.set_title("Edge Weight vs Congestion", fontweight="bold")ax1.legend()ax1.spines["top"].set_visible(False)ax1.spines["right"].set_visible(False)colors = ["#e74c3c" if h == "R1-A" else "#2ecc71" for h in hops]ax2.bar(range(len(factors)), [1]*len(factors), color=colors, edgecolor="black")ax2.set_xticks(range(len(factors)))ax2.set_xticklabels([f"{f:.1f}" for f in factors])ax2.set_xlabel("Congestion factor on R1-A")ax2.set_yticks([])ax2.set_title("Dijkstra's Routing Decision", fontweight="bold")ax2.legend(handles=[    mpatches.Patch(color="#e74c3c", label="Routes to R1-A"),    mpatches.Patch(color="#2ecc71", label="Routes to R1-B"),], loc="upper left")ax2.spines["top"].set_visible(False)ax2.spines["right"].set_visible(False)ax2.spines["left"].set_visible(False)plt.tight_layout()plt.show()print("\n[PASS] Congestion shifts Dijkstra: overloaded R1-A reroutes to R1-B")

---## Criterion 3: HC-2 Deterministic ReplayGraph routing adds dynamic state (congestion factors change as patientsmove through the system). We need to prove this doesn't breakdeterminism: same seed, same toggles, same output.Dijkstra on a weighted graph is deterministic for the same graph state.Same seed produces the same arrival order, which produces the samecongestion updates, which produces the same weights, which produces thesame paths. The chain is fully determined.

In [ ]:
# Run IRON BRIDGE 48h twice with same seedm1, e1 = run_iron_bridge(seed=42)m2, e2 = run_iron_bridge(seed=42)fields = ['total_arrivals', 'completed', 'in_system', 'outcomes']print(f"{'Field':<18} {'Run 1':<25} {'Run 2':<25} {'Match':>5}")print("-" * 75)all_ok = Truefor name in fields:    ok = m1[name] == m2[name]    if not ok:        all_ok = False    print(f"{name:<18} {str(m1[name]):<25} {str(m2[name]):<25} {'OK' if ok else 'FAIL':>5}")# Per-facility countsc1 = count_facility_arrivals(e1)c2 = count_facility_arrivals(e2)fac_ok = c1 == c2if not fac_ok:    all_ok = Falseprint(f"{'facility_counts':<18} {'(see below)':<25} {'(see below)':<25} {'OK' if fac_ok else 'FAIL':>5}")for fid in sorted(c1.keys()):    print(f"  {fid:<16} {c1[fid]:<25} {c2[fid]:<25}")assert all_ok, "HC-2 VIOLATION"print("\n[PASS] HC-2: deterministic replay — two runs are byte-identical")

---## Criterion 4: KL-6 DISPOSITION InvariantThe foundational accounting rule: every patient that enters the systemmust either complete their journey (DISPOSITION) or still be in transit(in_system) at sim end. No patients lost, no double-counting.```arrivals == dispositions + in_system```

In [ ]:
# KL-6 on the IRON BRIDGE graph routing rundisps = len(engine_ib.event_store.events_of_type("DISPOSITION"))arr = m_ib["total_arrivals"]in_sys = m_ib["in_system"]ok = arr == disps + in_sysassert ok, f"KL-6 VIOLATION: {arr} != {disps} + {in_sys}"assert arr > 0print(f"  {arr} arrivals == {disps} dispositions + {in_sys} in_system")print(f"\n  Outcome breakdown:")for outcome, count in sorted(m_ib["outcomes"].items()):    pct = 100 * count / m_ib["completed"] if m_ib["completed"] > 0 else 0    print(f"    {outcome}: {count} ({pct:.1f}%)")print("\n[PASS] KL-6: DISPOSITION invariant holds")

---## Exit SummaryAll four criteria are assertion-gated — if you see the summary below,every cell passed without exception.

In [ ]:
# Outcome pie chartoutcome_keys = sorted(m_ib["outcomes"].keys())outcome_vals = [m_ib["outcomes"][k] for k in outcome_keys]outcome_colors = {"DECEASED": "#e74c3c", "RTD": "#2ecc71", "STRATEVAC": "#3498db"}fig, ax = plt.subplots(figsize=(6, 4))ax.pie(outcome_vals, labels=outcome_keys, autopct="%1.0f%%",       colors=[outcome_colors.get(k, "#999") for k in outcome_keys],       startangle=90, textprops={"fontsize": 11})ax.set_title("IRON BRIDGE 48h Outcomes (Graph Routing)", fontweight="bold")plt.tight_layout()plt.show()# Summaryprint("=" * 65)print("PHASE 1.5 — GRAPH ROUTING EXIT CRITERIA")print("=" * 65)criteria = [    ("1. Both R1 nodes active",  f"ALPHA={graph_counts['R1-ALPHA']}, BRAVO={graph_counts['R1-BRAVO']}, imbalance={imbalance:.3f}"),    ("2. Congestion shifts route", "Overloaded R1-A reroutes to R1-B"),    ("3. HC-2 deterministic",      "Two 48h runs, byte-identical"),    ("4. KL-6 invariant",          f"{arr}=={disps}+{in_sys}"),]for name, detail in criteria:    print(f"  [PASS] {name}")    print(f"         {detail}")print("=" * 65)print()print("DECISION: GO for Engine Room demo")print()print("What changed:")print("  - routing.py: Dijkstra replaces role-walk (toggle-gated)")print("  - engine.py: congestion wired at FACILITY_ARRIVAL + DISPOSITION")print("  - topology.py: get_travel_time() returns base_time (not weight)")print("  - iron_bridge.yaml: 5-node LSCO preset restored")print()print("What didn't change:")print("  - Toggle OFF: identical to before on all topologies")print("  - Clinical short-circuits: T3 RTD, T4 stays — untouched")print("  - SimPy yield points: still only in engine.py")print("  - Zero SimPy imports in routing.py")